# DenseNet121 Training Pipeline: MixUp, TTA & Advanced Scheduling

## 1. Research Goal
This notebook represents our advanced experiments with the **DenseNet121** architecture. While ResNet18 provided a strong baseline, DenseNet's feature reuse mechanism theoretically allows for more parameter-efficient learning. 

**Key Innovations in Experiment v21:**
1.  **MixUp Augmentation**: To improve convex behavior of the predictions and reduce overfitting.
2.  **Weighted Huber Loss**: To handle outliers in the biomass ground truth without the harsh penalties of MSE.
3.  **Hybrid Scheduling**: Rapid convergence with OneCycleLR followed by fine-tuning with ReduceLROnPlateau.
4.  **Test Time Augmentation (TTA)**: Averaging predictions across horizontal and vertical flips.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from typing import Optional, Iterable, Tuple
import joblib

TARGET_COLUMNS = ["Dry_Clover_g", "Dry_Dead_g", "Dry_Green_g", "Dry_Total_g", "GDM_g"]
# Weights adjusted based on feature importance/difficulty
TARGET_WEIGHTS = {"Dry_Clover_g": 0.1, "Dry_Dead_g": 0.1, "Dry_Green_g": 0.1, "Dry_Total_g": 0.5, "GDM_g": 0.2}

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Loss Function: Weighted Huber Loss

Mean Squared Error (MSE) is sensitive to outliers. In biomass estimation, some samples might have extreme values due to measurement errors or natural variability.

**Huber Loss** acts as MSE for small errors and Mean Absolute Error (MAE) for large errors, providing a robust compromise. We further weight the loss to prioritize the aggregate target `Dry_Total_g`.

In [ ]:
class WeightedHuberLoss(nn.Module):
    def __init__(self, device, target_weights, beta=1.0):
        super().__init__()
        self.register_buffer("weights", torch.tensor(target_weights, dtype=torch.float32, device=device))
        self.huber = nn.SmoothL1Loss(reduction="none", beta=beta)

    def forward(self, preds, targets):
        loss_per_elem = self.huber(preds, targets)          
        weighted = loss_per_elem * self.weights             
        return weighted.mean()

## 3. Regularization Strategy: MixUp

We implement **MixUp** [Zhang et al., 2017](https://arxiv.org/abs/1710.09412), which trains the network on linear combinations of pairs of examples and their labels. 

$$ \tilde{x} = \lambda x_i + (1-\lambda) x_j $$
$$ \tilde{y} = \lambda y_i + (1-\lambda) y_j $$

This forces the model to favor simple linear behavior in-between training examples.

In [ ]:
def mixup_data(x: torch.Tensor, y: torch.Tensor, alpha: float = 0.4):
    if alpha is None or alpha <= 0 or x.size(0) < 2:
        return x, y
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x_mix = lam * x + (1.0 - lam) * x[idx]
    y_mix = lam * y + (1.0 - lam) * y[idx]
    return x_mix, y_mix

## 4. Models & Training Loop
We integrate **Automatic Mixed Precision (AMP)** to speed up training on modern GPUs and reduce memory usage, allowing for larger batch sizes.

In [ ]:
def build_model(architecture: str = "densenet121", num_targets: int = 5, weights=None):
    model = models.densenet121(weights=weights)
    model.classifier = nn.Linear(model.classifier.in_features, num_targets)
    return model

def train_regression(model, num_epochs, train_dl, valid_dl, loss_fn, optimizer, device, 
                     scheduler=None, plateau_scheduler=None, explore_epochs=0, 
                     mixup_fn=None, use_amp=False, patience=10, ckpt_path=None):
    # ... [Training loop same as previous implementation] ...
    # Hidden for brevity in this documentation. The logic handles AMP scaling,
    # MixUp application, and dual-scheduler stepping.
    pass # (Placeholder for the actual training function code seen in v21)

In [ ]:
# Re-inserting the full training code block from v21 for reproducibility
def train_regression(model, num_epochs, train_dl, valid_dl, loss_fn, optimizer, device, 
                     scheduler=None, plateau_scheduler=None, explore_epochs=0, 
                     mixup_fn=None, use_amp=False, patience=2, min_epochs_before_es=0, ckpt_path=None):
    loss_hist_train, loss_hist_valid = [], []
    use_cuda_amp = (use_amp and device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_cuda_amp)
    best_val, bad_epochs = float("inf"), 0

    for epoch in range(num_epochs):
        model.train()
        running = 0.0
        for x_batch, y_batch in train_dl:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            if mixup_fn: x_batch, y_batch = mixup_fn(x_batch, y_batch)
            optimizer.zero_grad(set_to_none=True)
            if use_cuda_amp:
                with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                    pred = model(x_batch)
                    loss = loss_fn(pred, y_batch)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = loss_fn(model(x_batch), y_batch)
                loss.backward()
                optimizer.step()
            if scheduler and (explore_epochs == 0 or epoch < explore_epochs): scheduler.step()
            running += loss.item()
        
        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for x_batch, y_batch in valid_dl:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                with torch.amp.autocast("cuda", enabled=use_cuda_amp):
                     val_running += loss_fn(model(x_batch), y_batch).item()
        
        train_loss, val_loss = running/len(train_dl), val_running/len(valid_dl)
        print(f"Epoch {epoch+1}| Train: {train_loss:.4f} | Val: {val_loss:.4f}")
        
        if plateau_scheduler and epoch >= explore_epochs: plateau_scheduler.step(val_loss)
        
        if val_loss < best_val - 1e-6:
            best_val, bad_epochs = val_loss, 0
            if ckpt_path: torch.save(model.state_dict(), ckpt_path)
        elif epoch >= min_epochs_before_es:
            bad_epochs += 1
            if bad_epochs >= patience: 
                print("Early Stopping")
                break

## 5. Execution: 5-Fold Cross Validation
We execute 5-Fold CV using a hybrid scheduling strategy:
- **Phase 1 (Epochs 0-25)**: `OneCycleLR` to rapidly define a good basin of attraction.
- **Phase 2 (Epochs 25+)**: `ReduceLROnPlateau` for fine-grained convergence.

In [ ]:
DEVICE = get_device()
BASE_DIR = "/kaggle/input/csiro-biomass"
TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")
LR = 1e-4
EPOCHS = 160
EXPLORE_EPOCHS = 25

# ... [Setup code for DataLoaders and KFold loop remains similar to original] ...